# Ray Distributed Model & Dataset Download

This notebook runs the **distributed download pipeline** from `app/scale_train/train.py` on an Anyscale workspace.

**What it does:**
1. Connects to the Ray cluster on the Anyscale workspace
2. Downloads a pre-trained model (Mistral-7B) to shared storage (`/mnt/cluster_storage`)
3. Distributes the download of a large dataset (Fineweb-Edu) across multiple workers in parallel
4. Validates all downloads

**Prerequisites:**
- Anyscale workspace `anyscale_workspace` is RUNNING
- PVC `blob-pvc-checkpoint` is Bound in `anyscale-operator` namespace
- `/mnt/cluster_storage` is mounted via Anyscale cloud `file_storage`
- `HF_TOKEN` env var is set (required for gated models like Mistral-7B)

## 1. Install Dependencies
Ensure required packages are available in the workspace environment.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "pyarrow", "pandas", "ray[data]", "requests", "tqdm",
    "torch", "transformers", "accelerate", "tokenizers", "sentencepiece", "protobuf"
])

## 2. Configuration & Constants

Define storage paths and model/dataset identifiers. The shared volume is mounted at `/mnt/cluster_storage` by the Anyscale cloud's `file_storage` PVC.

In [ ]:
import os
import time
import sys
import ray
from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files

# ---- Storage Path ----
STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/cluster_storage")

# ---- Model Config ----
MODEL_SAVE_PATH = os.path.join(STORAGE_PATH, "model")
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # gated model - needs HF_TOKEN

# ---- Dataset Config ----
DATASET_ID = "HuggingFaceFW/fineweb-edu"
DATASET_SUBSET = "sample/100BT"  # ~460 parquet files
DATASET_SAVE_PATH = os.path.join(STORAGE_PATH, "dataset")

# ---- Checkpoint Config ----
CHECKPOINT_SAVE_PATH = os.path.join(STORAGE_PATH, "checkpoint")

print(f"Storage path:    {STORAGE_PATH}")
print(f"Model save path: {MODEL_SAVE_PATH}")
print(f"Dataset path:    {DATASET_SAVE_PATH}")
print(f"Checkpoint path: {CHECKPOINT_SAVE_PATH}")
print(f"Model ID:        {MODEL_ID}")
print(f"Dataset ID:      {DATASET_ID} ({DATASET_SUBSET})")

## 3. Validate & Setup Storage

Verify that `/mnt/cluster_storage` is mounted (PVC is working) and create required subdirectories.

In [ ]:
def validate_and_setup_storage():
    """Validates shared storage path exists and creates subdirectories."""
    print(f"[Init] Checking storage path: {STORAGE_PATH}")

    if not os.path.exists(STORAGE_PATH):
        raise FileNotFoundError(
            f"[CRITICAL] Storage path {STORAGE_PATH} not found! "
            f"Ensure PV is mounted and Anyscale cloud has file_storage configured."
        )

    print(f"[Init] ✅ Storage path confirmed: {STORAGE_PATH}")

    # Create required subdirectories
    for path, label in [(MODEL_SAVE_PATH, "model"), (DATASET_SAVE_PATH, "dataset"), (CHECKPOINT_SAVE_PATH, "checkpoint")]:
        os.makedirs(path, exist_ok=True)
        print(f"[Init] ✅ Created/verified: {path}")

validate_and_setup_storage()

## 4. Initialize Ray Cluster

Connect to the Ray cluster running on the Anyscale workspace. Log available resources.

In [ ]:
if ray.is_initialized():
    print("[Init] Ray is already initialized.")
else:
    ray.init(address="auto")

resources = ray.available_resources()
print(f"[Init] ✅ Connected to Ray cluster")
print(f"  CPUs:  {resources.get('CPU', 0)}")
print(f"  GPUs:  {resources.get('GPU', 0)}")
print(f"  Nodes: {len(ray.nodes())}")
print(f"  Memory: {resources.get('memory', 0) / 1e9:.1f} GB")

## 5. Task 1 — Download Model (Centralized)

Download the Mistral-7B model to shared storage. Runs as a single Ray remote task to offload memory from the head node.

In [ ]:
@ray.remote(num_cpus=1)
def download_model():
    """Downloads the model repository to shared storage."""
    import os
    from huggingface_hub import snapshot_download

    print(f"[Model] Downloading {MODEL_ID} to {MODEL_SAVE_PATH}...")

    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        print(f"[Model] HF_TOKEN: {hf_token[:4]}...{hf_token[-4:]}")
    else:
        print("[Model] WARNING: HF_TOKEN not set — gated models will fail")

    token_arg = hf_token if hf_token else False

    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_SAVE_PATH,
        token=token_arg,
        local_dir_use_symlinks=False,
    )
    print(f"[Model] ✅ Model saved to {MODEL_SAVE_PATH}")


print("[Step 1] Downloading model...")
ray.get(download_model.remote())
print("[Step 1] ✅ Model download complete")

## 6. Task 2 — Download Dataset (Distributed via Ray Data)

Define the worker function that downloads batches of dataset files. Each worker downloads files from Hugging Face Hub to the shared storage.

In [ ]:
def download_file_wrapper(batch):
    """
    Ray Data map function: Downloads a batch of dataset files.
    Ray Data passes a dictionary-like batch (e.g. {'item': [file1, file2]}).
    """
    import os
    import ray
    from huggingface_hub import hf_hub_download

    worker_id = f"{ray.get_runtime_context().get_task_id()}"
    node_ip = ray.util.get_node_ip_address()
    worker_tag = f"Worker {worker_id[-8:]}@{node_ip}"

    try:
        filenames = batch.get("item", [])
    except AttributeError:
        filenames = batch

    file_names = []
    statuses = []

    token_arg = os.environ.get("HF_TOKEN")
    if not token_arg:
        token_arg = False

    print(f"[{worker_tag}] Starting batch of {len(filenames)} files")

    for filename in filenames:
        try:
            file_path = os.path.join(DATASET_SAVE_PATH, filename)

            if os.path.exists(file_path):
                print(f"[{worker_tag}] Skipped (exists): {filename}")
                file_names.append(filename)
                statuses.append("skipped")
                continue

            hf_hub_download(
                repo_id=DATASET_ID,
                filename=filename,
                repo_type="dataset",
                local_dir=DATASET_SAVE_PATH,
                token=token_arg
            )
            print(f"[{worker_tag}] Downloaded: {filename}")
            file_names.append(filename)
            statuses.append("downloaded")
        except Exception as e:
            print(f"[{worker_tag}] Failed: {filename} - {e}")
            file_names.append(filename)
            statuses.append("failed")

    print(f"[{worker_tag}] Batch complete: {len(file_names)} files processed")
    return {"file": file_names, "status": statuses}

print("✅ download_file_wrapper defined")

## 7. Task 3 — Orchestrate Distributed Dataset Download

List files from the Hugging Face repo, filter to the target subset, create a Ray Dataset, and distribute the download work across the cluster using `map_batches`.

- `batch_size=20` → 20 files per task
- `num_cpus=4` per task
- `concurrency=20` → 20 parallel tasks across the cluster

In [ ]:
def download_dataset_distributed():
    """Orchestrates distributed download of dataset files using Ray Data."""
    print(f"[Dataset] Fetching file list for {DATASET_ID} ({DATASET_SUBSET})...")

    try:
        all_files = list_repo_files(
            repo_id=DATASET_ID,
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN")
        )
    except Exception as e:
        print(f"[Dataset] Error listing repo files: {e}")
        return

    print(f"[Dataset] Total files in repo: {len(all_files)}")

    target_files = [f for f in all_files if f.endswith(".parquet") and DATASET_SUBSET in f]
    print(f"[Dataset] Filtered to {len(target_files)} parquet files matching '{DATASET_SUBSET}'")
    print(f"[Dataset] Triggering distributed download via Ray Data...")

    start_time = time.time()

    input_data = [{"item": f} for f in target_files]
    ds = ray.data.from_items(input_data)

    downloaded_ds = ds.map_batches(
        download_file_wrapper,
        batch_size=20,
        num_cpus=4,
        concurrency=20
    )

    all_results = downloaded_ds.take_all()
    duration = time.time() - start_time

    success_count = sum(1 for r in all_results if r["status"] != "failed")
    failed_count = sum(1 for r in all_results if r["status"] == "failed")

    print(f"\n[Dataset] Summary: Processed {len(all_results)} files")
    print(f"[Dataset] ✅ Downloaded {success_count}/{len(target_files)} files in {duration:.2f}s")
    if failed_count > 0:
        failed_files = [r["file"] for r in all_results if r["status"] == "failed"]
        print(f"[Dataset] ⚠️ {failed_count} files failed: {failed_files}")


print("[Step 2] Downloading dataset (distributed)...")
download_dataset_distributed()
print("[Step 2] ✅ Dataset download complete")

## 8. Validate Downloads

Verify that model and dataset files are present in shared storage.

In [ ]:
def validate_downloads():
    """Verifies model and dataset files are present in shared storage."""
    print("[Validation] Verifying downloads...")
    validation_failed = False

    # ---- Validate Model ----
    if os.path.exists(MODEL_SAVE_PATH):
        model_files = os.listdir(MODEL_SAVE_PATH)
        if "config.json" in model_files:
            print(f"[Validation] ✅ Model: Found config.json + {len(model_files)-1} other files in {MODEL_SAVE_PATH}")
        elif len(model_files) > 0:
            print(f"[Validation] ⚠️ Model: Directory not empty but config.json missing. Files: {len(model_files)}")
        else:
            print(f"[Validation] ❌ Model: Directory {MODEL_SAVE_PATH} is empty")
            validation_failed = True
    else:
        print(f"[Validation] ❌ Model: Directory {MODEL_SAVE_PATH} does not exist")
        validation_failed = True

    # ---- Validate Dataset ----
    if os.path.exists(DATASET_SAVE_PATH):
        parquet_files = []
        for root, dirs, files in os.walk(DATASET_SAVE_PATH):
            for f in files:
                if f.endswith(".parquet"):
                    parquet_files.append(os.path.relpath(os.path.join(root, f), DATASET_SAVE_PATH))

        if len(parquet_files) > 0:
            print(f"[Validation] ✅ Dataset: Found {len(parquet_files)} parquet files under {DATASET_SAVE_PATH}")
            # Show first 5
            for pf in parquet_files[:5]:
                print(f"  - {pf}")
            if len(parquet_files) > 5:
                print(f"  ... and {len(parquet_files) - 5} more")
        else:
            print(f"[Validation] ❌ Dataset: No parquet files found under {DATASET_SAVE_PATH}")
            validation_failed = True
    else:
        print(f"[Validation] ❌ Dataset: Directory {DATASET_SAVE_PATH} does not exist")
        validation_failed = True

    if validation_failed:
        print("\n❌ One or more validations failed!")
        return False
    else:
        print("\n✅ All validations passed!")
        return True


print("[Step 3] Validating downloads...")
validate_downloads()

## 9. Summary — Check Shared Storage

List the contents of `/mnt/cluster_storage` to confirm everything landed correctly.

In [ ]:
print(f"\n📁 Contents of {STORAGE_PATH}:")
print("=" * 60)

for entry in sorted(os.listdir(STORAGE_PATH)):
    full_path = os.path.join(STORAGE_PATH, entry)
    if os.path.isdir(full_path):
        sub_items = os.listdir(full_path)
        print(f"  📂 {entry}/ ({len(sub_items)} items)")
        for sub in sorted(sub_items)[:5]:
            sub_full = os.path.join(full_path, sub)
            size = os.path.getsize(sub_full) if os.path.isfile(sub_full) else "dir"
            print(f"       {sub}  ({size})")
        if len(sub_items) > 5:
            print(f"       ... and {len(sub_items) - 5} more")
    else:
        size = os.path.getsize(full_path)
        print(f"  📄 {entry}  ({size} bytes)")

print("\n✅ Done!")